# PubMed fetch - run this locally

Steps 0.6 / 1.4 of `PLAN.md`, on your machine instead of Kaggle.

Local is the better home for this job:

| | Local | Kaggle |
|---|---|---|
| GPU needed | no - pure network I/O | no, so a GPU session wastes quota |
| Session limit | none | 12 h, and `/kaggle/working` is lost when it ends |
| Getting the file back | already in `data/` | needs Save Version, then a download |

**Kernel:** select **ADE-Sentinel (.venv)** in the top-right, not a global Python.

Expect roughly 40 minutes with an API key, measured. The job is resumable: each
year is written to its own shard and completed years are skipped on a rerun, so
an interrupted kernel costs you only the year in flight.


## 1. Check the API key

The key lives in the `NCBI_API_KEY` user environment variable, not in this
notebook - this file is committed to a public repo.

If this prints `MISSING`, **VS Code was started before the variable was set**.
Close VS Code completely and reopen it; a reloaded window is not enough, because
the environment is inherited from the process that launched it.


In [ ]:
import os

key = os.environ.get('NCBI_API_KEY', '').strip()
if key:
    print(f'NCBI_API_KEY: present ({len(key)} chars, {key[:6]}...) -> 10 req/s')
else:
    print('NCBI_API_KEY: MISSING -> 3 req/s, roughly 3x slower')
    print('Restart VS Code, or uncomment the line below for this session only.')

# One-session fallback. Do NOT commit a key here:
# os.environ['NCBI_API_KEY'] = 'paste-key-here'


## 2. Import the fetcher

`fetch_pubmed` reads the key lazily on every request, so cell order does not
matter - setting the key above still takes effect even though the import
happens after it.


In [ ]:
import sys, time, pathlib

REPO = pathlib.Path.cwd()
if REPO.name == 'notebooks':
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from scripts.fetch_pubmed import (
    QUERY, MIN_YEAR, MAX_YEAR, PER_YEAR_CAP, SHARD_DIR, DEFAULT_OUT,
    api_key, count_hits, fetch_year, merge_shards,
)

print('repo      :', REPO)
print('query     :', QUERY)
print('years     :', MIN_YEAR, '-', MAX_YEAR)
print('cap/year  :', f'{PER_YEAR_CAP:,}')
print('rate      :', '10 req/s' if api_key() else '3 req/s')
print('shards    :', SHARD_DIR)
print('output    :', DEFAULT_OUT)


## 3. Size the job

The query matches far more than we need, which is the point of the per-year
cap: it samples a temporally balanced slice rather than taking whichever years
happen to be biggest. Because `hasabstract` is part of the query itself, every
PMID fetched has an abstract - measured retention is 100%, so the cap is the
yield.


In [ ]:
years = MAX_YEAR - MIN_YEAR + 1
matched = count_hits(f'{MIN_YEAR}/01/01', f'{MAX_YEAR}/12/31')

print(f'{matched:,} records match the query across {years} years')
print(f'capped at {PER_YEAR_CAP:,}/year -> target ~{PER_YEAR_CAP * years:,}')
print(f'measured 9,174 for 2011, so expect ~{9174 * years:,} abstracts')

done = sorted(p.stem for p in SHARD_DIR.glob('*.jsonl')) if SHARD_DIR.exists() else []
print(f'\nyears already fetched: {len(done)}', f'({done[0]}-{done[-1]})' if done else '')


## 4. Fetch

Runs year by year, printing as it goes. Each year is split into 12 month
windows because the union query exceeds NCBI's 9,998-per-search cap - that is
the normal path here, not an error.

Interrupting the kernel is safe: a shard
is written to a `.partial` file and renamed only once the year is complete, so a
half-finished year is never mistaken for a done one.

Rerun this cell after an interruption and it resumes where it stopped.


In [ ]:
SHARD_DIR.mkdir(parents=True, exist_ok=True)
started = time.time()

for year in range(MIN_YEAR, MAX_YEAR + 1):
    shard = SHARD_DIR / f'{year}.jsonl'
    if shard.exists():
        print(f'  {year}: done, skipping')
        continue
    fetch_year(year, shard)

print(f'\nfetch wall clock: {(time.time() - started) / 60:.1f} min')


## 5. Merge the shards and check the floor


In [ ]:
total_records, duplicates = merge_shards(DEFAULT_OUT)

size_mb = DEFAULT_OUT.stat().st_size / 1e6
print(f'{total_records:,} unique abstracts -> {DEFAULT_OUT}')
print(f'{duplicates:,} cross-year duplicate PMIDs dropped')
print(f'{size_mb:,.1f} MB on disk')

if total_records < 100_000:
    print(f'\n[!!] below the 100k floor in PRD 6.2.')
    print('     Widen MIN_YEAR, or add the subheading clauses from the PRD query')
    print('     table - month windows kick in automatically past the 9,998 cap.')
else:
    print(f'\n[ok] clears the 100k floor.')


## 6. Eyeball the result

Confirm the records look like biomedical abstracts and that nested markup
survived - a title ending mid-phrase means the `<i>`/`<sub>` flattening broke.


In [ ]:
import json

with DEFAULT_OUT.open(encoding='utf-8') as fh:
    for i, line in enumerate(fh):
        if i >= 3:
            break
        rec = json.loads(line)
        print(f"PMID {rec['pmid']}")
        print(f"  title    : {rec['title'][:110]}")
        print(f"  abstract : {rec['abstract'][:180]}...")
        print()


---

## Next

1. Record the record count, the wall-clock time from cell 4, the exact query and
   the date range in `report/data_documentation.md` (step 1.5). This is an
   explicit instructor requirement, not bookkeeping.
2. `data/pubmed_corpus.jsonl` is git-ignored by design - it goes to the versioned
   Kaggle Dataset via `scripts/push_to_kaggle.py` when Phase 3 needs it remotely.
3. Phase 2.3 sentence-splits this corpus; Phase 3 trains Word2Vec and FastText on
   it. Both are CPU-only, so both can also run here.
